# Phase B — Parameter Sweep on Variant B

Run a 1D parameter sweep around variant B (B-spline + SCReLU base) to find the best
`(grid_size, spline_order)` combination. Single seed per cell — broad signal hunt.

| Cell | grid_size | spline_order | Rationale |
|------|-----------|--------------|-----------|
| `kan_variant_b`     (baseline) | 5  | 3 | Phase A winner |
| `kan_variant_b_g3`  | 3  | 3 | Less capacity |
| `kan_variant_b_g7`  | 7  | 3 | More capacity |
| `kan_variant_b_g10` | 10 | 3 | Too-many-knots stress test |
| `kan_variant_b_o2`  | 5  | 2 | Quadratic basis (better quantization) |
| `kan_variant_b_o4`  | 5  | 4 | Quartic basis (smoother) |

**Plan**: 6 runs × 1 seed × ~10 min on T4 ≈ 1 hour total.

**Decision rule**: rank by final running loss. Per-variant stdev from Phase A seed validation
was ~9e-05 (variant B). If any cell beats baseline by > 2× stdev (~1.8e-04), the signal is
real — expand into a 2D cell on that axis. Otherwise stop the param sweep, move to Phase C.

**Runtime**: GPU (T4 or better).


## 1. Install Rust + clone repo

In [ ]:
%%bash
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version

In [ ]:
%%bash
set -e
if [ -d /content/bullet ]; then
    cd /content/bullet
    git fetch origin
    git reset --hard origin/main
else
    cd /content
    git clone https://github.com/y0sif/bullet.git
    cd bullet
fi
git log -1 --oneline

## 2. Download training data (test77 binpack)

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test77.binpack ]; then
    echo "Downloading test77 binpack from HuggingFace (~1.3 GB compressed)..."
    wget -q -O test77.binpack.zst \
        "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst"
    echo "Download complete. Decompressing..."
    zstd -d test77.binpack.zst -o test77.binpack --rm
    echo "Done!"
fi

ls -lh test77.binpack

## 3. Build the 6 Phase B variants

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet
for v in b b_g3 b_g7 b_g10 b_o2 b_o4; do
    echo "=== Building kan_variant_${v} ==="
    cargo build --release --example kan_variant_${v} 2>&1 | tail -3
done

## 4. Run all 6 variants (single seed each)

Each run writes to `/content/variant_<v>_log.txt`. Checkpoints are wiped between runs
so per-variant artefacts don't accumulate (we only need the running-loss values from logs).

In [ ]:
import subprocess, sys, os, shutil

os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Order: baseline first, then sweeps
VARIANTS = ["b", "b_g3", "b_g7", "b_g10", "b_o2", "b_o4"]

# Set RESET_LOGS=True to force re-running even if a log already exists
RESET_LOGS = False

for v in VARIANTS:
    log_path = f"/content/variant_{v}_log.txt"
    if os.path.exists(log_path) and not RESET_LOGS:
        print(f"SKIP {log_path} (already exists; set RESET_LOGS=True to rerun)")
        continue

    ckpt_dir = "/content/bullet/checkpoints"
    if os.path.isdir(ckpt_dir):
        shutil.rmtree(ckpt_dir, ignore_errors=True)

    print(f"\n{'='*70}\n variant={v}  ->  {log_path}\n{'='*70}")
    cmd = ["cargo", "run", "--release", "--example", f"kan_variant_{v}"]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd="/content/bullet", text=True, bufsize=1,
    )
    with open(log_path, "w") as log:
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            log.write(line)
    proc.wait()
    print(f"\nExit code: {proc.returncode}")
    if proc.returncode != 0:
        print("FAILED — stopping. Fix the issue before continuing.")
        raise SystemExit(1)

## 5. Rank, plot, decide

Computes the final running loss for each variant. Phase A seed validation measured
variant-B per-seed stdev ≈ 9e-05; treat ~1.8e-04 (2× stdev) as the
"real signal" threshold for this single-seed sweep.

In [ ]:
import re
import matplotlib.pyplot as plt

def strip_ansi(s):
    return re.sub(r'\x1b\[[0-9;]*m', '', s)

def parse_bullet_log(path):
    losses = []
    with open(path) as f:
        for line in f:
            m = re.search(r'superbatch\s+(\d+)\s+\|.*?running loss\s+([\d.]+)', strip_ansi(line))
            if m:
                losses.append((int(m.group(1)), float(m.group(2))))
    return losses

VARIANTS = ["b", "b_g3", "b_g7", "b_g10", "b_o2", "b_o4"]
LABELS = {
    "b":     "B baseline   (g=5,  o=3)",
    "b_g3":  "B-g3         (g=3,  o=3)",
    "b_g7":  "B-g7         (g=7,  o=3)",
    "b_g10": "B-g10        (g=10, o=3)",
    "b_o2":  "B-o2         (g=5,  o=2)",
    "b_o4":  "B-o4         (g=5,  o=4)",
}

# Per-variant stdev floor from Phase A seed validation
PHASE_A_STDEV = 8.95e-05
SIGNAL_THRESHOLD = 2 * PHASE_A_STDEV  # ~1.8e-04

curves = {}
finals = {}
for v in VARIANTS:
    path = f"/content/variant_{v}_log.txt"
    try:
        losses = parse_bullet_log(path)
    except FileNotFoundError:
        print(f"Missing {path}")
        continue
    if not losses:
        print(f"No loss lines in {path}")
        continue
    curves[v] = losses
    finals[v] = losses[-1][1]

# Print table sorted by final loss
print(f"{'Variant':<28}  {'final loss':>12}  {'Δ vs baseline':>14}  {'>2× stdev?':>12}")
print("-" * 72)
baseline_loss = finals.get("b")
ranked = sorted(finals.items(), key=lambda kv: kv[1])
for v, loss in ranked:
    if baseline_loss is None:
        delta_str, signal = "—", ""
    else:
        delta = loss - baseline_loss
        delta_str = f"{delta:+.6f}"
        if v == "b":
            signal = "(baseline)"
        elif delta < -SIGNAL_THRESHOLD:
            signal = "YES (better)"
        elif delta > SIGNAL_THRESHOLD:
            signal = "YES (worse)"
        else:
            signal = "no (tied)"
    marker = "  <-- best" if v == ranked[0][0] else ""
    print(f"{LABELS[v]:<28}  {loss:.6f}  {delta_str:>14}  {signal:>12}{marker}")

# Decision summary
if baseline_loss is not None:
    print(f"\nSignal threshold: |Δ| > {SIGNAL_THRESHOLD:.6f}  (= 2 × Phase-A stdev {PHASE_A_STDEV:.2e})")
    winners = [v for v, loss in finals.items() if v != "b" and loss < baseline_loss - SIGNAL_THRESHOLD]
    if winners:
        print(f"\nReal-signal winners over baseline: {winners}")
        print("  -> Expansion trigger met. Run a 2D cell on the winning axis.")
    else:
        print("\nNo cell beats baseline by > 2× stdev. Stop param sweep, move to Phase C (topology).")

# Plot 1: overlaid loss curves
fig, ax = plt.subplots(figsize=(11, 7))
colors = {"b": "k", "b_g3": "C0", "b_g7": "C1", "b_g10": "C3", "b_o2": "C2", "b_o4": "C4"}
for v in VARIANTS:
    if v not in curves: continue
    losses = curves[v]
    lw = 2.5 if v == "b" else 1.8
    ax.plot([x[0] for x in losses], [x[1] for x in losses],
            color=colors.get(v, "gray"), linewidth=lw, label=LABELS[v])
ax.set_xlabel("Superbatch")
ax.set_ylabel("Running loss")
ax.set_title("Phase B — Parameter Sweep on Variant B")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/content/phase_b_curves.png", dpi=150)
plt.show()

# Plot 2: final loss bar chart
fig, ax = plt.subplots(figsize=(10, 5))
xs = [LABELS[v] for v, _ in ranked]
ys = [loss for _, loss in ranked]
bar_colors = ["tab:gray" if v == "b" else ("tab:green" if loss < baseline_loss - SIGNAL_THRESHOLD else
                                             ("tab:red" if loss > baseline_loss + SIGNAL_THRESHOLD else "tab:blue"))
              for v, loss in ranked] if baseline_loss is not None else ["tab:blue"] * len(ranked)
ax.bar(range(len(xs)), ys, color=bar_colors)
ax.set_xticks(range(len(xs)))
ax.set_xticklabels(xs, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Final running loss")
ax.set_title("Phase B — Final Loss by Variant (sorted)")
if baseline_loss is not None:
    ax.axhline(baseline_loss, color="black", linestyle="--", linewidth=1, alpha=0.6, label=f"baseline {baseline_loss:.6f}")
    ax.axhspan(baseline_loss - SIGNAL_THRESHOLD, baseline_loss + SIGNAL_THRESHOLD,
               color="black", alpha=0.08, label=f"±2× stdev band")
    ax.legend(fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
ax.set_ylim(min(ys) * 0.998, max(ys) * 1.002)
plt.tight_layout()
plt.savefig("/content/phase_b_finals.png", dpi=150)
plt.show()

## 6. Save logs + plots to Drive (optional)

In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/kanue/phase_b_param_sweep'
os.makedirs(dest, exist_ok=True)

for v in ["b", "b_g3", "b_g7", "b_g10", "b_o2", "b_o4"]:
    src = f"/content/variant_{v}_log.txt"
    if os.path.exists(src):
        shutil.copy(src, dest)

for png in ["/content/phase_b_curves.png", "/content/phase_b_finals.png"]:
    if os.path.exists(png):
        shutil.copy(png, dest)

print(f"Saved to: {dest}")